# Vignette 02: Workflow Tour

Complete mlsurv workflow demonstration: preprocessing presets, tuning, evaluation, bootstrap CIs, external validation, plotting, and model persistence.

*Runtime: ~5 minutes on a modern laptop.*

**Datasets**: GBSG2 (training; Schumacher et al. 1994) + Rotterdam (external validation; Royston & Altman 2013) — a cross-cohort breast cancer validation design.

In [ ]:
import os

import pandas as pd
from sksurv.datasets import load_gbsg2
from mlsurv import SurvivalLearner
from mlsurv.utils import to_structured_array, load_survival_data

# Results are written to ./output/
os.makedirs("output", exist_ok=True)

## Load GBSG2 dataset

GBSG2 (Schumacher et al. 1994) is a node-positive breast cancer trial cohort from the German Breast Cancer Study Group: 686 patients, 8 clinical features, 299 events.

In [ ]:
X, y_raw = load_gbsg2()
X = X.copy()
X['horTh'] = (X['horTh'] == 'yes').astype(int)
X['menostat'] = (X['menostat'] == 'Post').astype(int)
X['tgrade'] = X['tgrade'].map({'I': 1, 'II': 2, 'III': 3}).astype(int)
y = to_structured_array(y_raw)

# GBSG2 survival times are recorded in days. Convert to months.
DAYS_PER_MONTH = 365.25 / 12
y['time'] = y['time'] / DAYS_PER_MONTH

print(f"GBSG2: n={len(X)}, features={X.shape[1]}, events={y['event'].sum()}")
X.head(5)


## Setup with preprocessor presets

`SurvivalLearner.setup()` configures the full preprocessing pipeline in one call:

- **`imputer='knn'`** — K-nearest-neighbors imputation for missing values (better than mean/median for correlated clinical features)
- **`scaler='robust'`** — Median/IQR scaling that handles outliers without distortion
- **`feature_selector='topk'`** — Univariate Cox regression selects the top-*K* most prognostic features (auto-selects *K* by default)

The data is split into train/test sets — the split fraction is controlled by `test_fraction` (default `0.3`) — and a stratified k-fold CV scheme is created for model comparison (default `cv_folds=5`, configurable via `cv_folds` and `cv_strategy`).

After configuring the pipeline, `setup()` runs proportional-hazards and log-hazard linearity diagnostics on the training data and consolidates them into a single **model recommendations table**. The table classifies every model into *Recommended / Cautionary / Not recommended* tiers based on:

- Dataset size and events-per-variable (EPV)
- PH assumption violations (Cox-family models are demoted when features violate PH)
- Log-linearity violations (Cox-family models are demoted when continuous features are non-linear in the log-hazard)
- A high-dimensional rule that disqualifies unregularized Cox models when `n_features >= n_events`

The whole bundle is available afterward via the read-only `learner.model_recommendations` property (next cell).

In [ ]:
learner = SurvivalLearner(X, y)
learner.setup(
    test_fraction=0.3,
    cv_folds=5,
    stratify='event',
    imputer='knn',
    scaler='robust',
    feature_selector='topk',
    random_state=42,
    n_jobs=1,
    time_unit='months',  # times converted to months above; labels every time axis
);


### Inspect the model recommendations

`learner.model_recommendations` returns the same bundle that printed above — always populated after `setup()`. It mixes a **model-level** verdict with **feature-level** evidence:

- **Model verdict → the tiers.** Read `recommended` / `cautionary` / `not_recommended`. PH and linearity are *per-feature* tests (they name violating covariates, not models), so they're already folded into the tiers — there is no per-model "passes PH" flag.
- **Feature-level evidence → the test DataFrames.** `ph_test_result` / `linearity_test_result` hold the full per-feature results (the setup-time snapshot).

In [ ]:
rec = learner.model_recommendations           # always available after setup()

print("Recommended:", rec.recommended)
print("Cautionary :", rec.cautionary)
print("Not recommended:", rec.not_recommended)
print(f"EPV={rec.epv:.1f}, N={rec.n_samples}, events={rec.n_events}")

# Feature-level evidence: the full PH test DataFrame
rec.ph_test_result

## Train baseline models

**Vignette 01 (Quickstart)** demonstrated the convenience wrapper `run()` which conducts training, tuning, evaluation, and bootstrapping in a single call. Here, we present the granular approach for users interested in inspecting and finely controlling each workflow stage.

`learner.train()` fits the listed models with their default hyperparameters and cross-validates each one. Passing a list trains them all in one call and returns a `{name: ModelResult}` dict — the granular building block for a baseline comparison. The scores show which model family looks strongest before you invest in tuning.

No tuning happens here (`train()` defaults to `tune=False`), so these are honest untuned baselines. Test-set metrics and bootstrap CIs come later, from `evaluate()` and `bootstrap()`.

In [ ]:
baselines = learner.train(['coxph', 'coxnet', 'rsf', 'weibull_aft', 'gbsa'])

### C-index comparison

The Concordance Index (C-Index) measures how well a model is able to rank observations. It answers the question: how often are two randomly selected subjects correctly ordered by the model risk scores? C=0.50 indicates performance is no better than random. `learner.plot_metric()` can display any of the four scoring metrics (Harrell's C-Index, IPCW C-Index, Antolini's C^td, Integrated Brier Score) for either CV training metrics, test set metrics (point values of the test set), or the bootstrap test set distribution. Here we've only completed the train step so only `source="cv"` is available.

In [ ]:
learner.plot_metric(source="cv",metric=["harrell","ipcw"],style="forest")  
# other metric options: integrated_brier, ctd

## Granular workflow: tune select models

Tuning costs time, and it does not help every model equally. The granular workflow lets you select which models to tune.

`learner.tune('<name>')` tunes a single model and **overwrites it in place**. Under the hood mlsurv applies **EPV-aware dynamic search spaces** (hyperparameter bounds constrained by the events-per-variable ratio to curb overfitting on small data) and optimizes with Optuna's TPE sampler ([Optuna](https://optuna.org); Akiba et al. 2019).

Let's tune the **GBSA** model to see how it may benefit from hyperparameter optimization.

In [ ]:
learner.tune('gbsa')

# Compare CV concordance before (untuned baseline) vs after (tuned)
before = baselines['gbsa'].cv_metric_mean  # CV scoring metric set in setup(), default harrell
after = learner.models['gbsa'].cv_metric_mean
print(f"gbsa CV C-index: {before:.3f} -> {after:.3f} ({after - before:+.3f})")

### Revisit CV C-Index for the Tuned GBSA

In [ ]:
learner.plot_metric(source="cv",metric="harrell",title="CV C-Index with Tuned GBSA")

## Evaluate on the test set

Called with no arguments, `learner.evaluate()` computes test-set metrics (C-index, Brier score, IBS) plus permutation importance for **every** model in the learner — GBSA tuned, the rest at their default hyperparameters. Test set metrics are point values calculated from the full test set on models trained on the full train set.

In [ ]:
eval_times=[12, 18, 24, 30, 36, 48, 60]
eval_result = learner.evaluate(eval_times=eval_times)
learner.plot_metric(source="test",metric="harrell",style="bar")

### Feature importance (multi-model)

Grouped permutation importance bars (Breiman 2001) across all models. Permutation importance shows how much performance (Harrell's C-index) drops when each feature is shuffled. Features important across multiple models are the most robust predictors.

In [ ]:
learner.plot_importance()

In [ ]:
learner.plot_importance(style="heatmap")

## Bootstrap confidence intervals

Called with no arguments, `learner.bootstrap()` runs bootstrap resampling on the test set for every model in the learner, attaching a confidence interval to each performance estimate. Like `evaluate()`, it covers all models in one pass.

In [ ]:
learner.bootstrap(include_time_resolved=True,include_predictions=True)

### Concordance Index with bootstrap sampling distribution

In [ ]:
learner.plot_metric() # when available, bootstrap is the default source for plot_metric
# raincloud style (default) shows both the bootstrap mean/distribution AND the test set value

### Calibration plot

Predicted vs. observed survival probabilities at specified time points, one curve per model. Points on the diagonal indicate perfect calibration.

We pass `bin_strategy="quantile"`, which groups patients into equal-count bins by predicted survival probability (rather than equal-width bins across [0, 1]). This matters for a model like GBSA whose predicted probabilities are tightly clustered: equal-width bins would drop nearly every patient into one bin and collapse the curve to a single point, while quantile bins spread the patients out so each model shows a full curve. The trade-off is an uneven x-axis (and, for heavily tied predictions, possibly fewer than `n_bins` points).

In [ ]:
learner.plot_calibration(models=["coxph","gbsa","coxnet"], bin_strategy="quantile")

### Kaplan-Meier survival curves

Patients are stratified into risk groups by each model's predicted survival. A panel grid shows all models side-by-side. Well-separated curves indicate the model meaningfully distinguishes high- vs low-risk patients.

In [ ]:
learner.plot_survival(models=["coxph","gbsa"])

### Time-dependent ROC curve 

Receiver operating characteristic (ROC) curve at specified time points, weighted by inverse probability of censoring (IPCW). Each model's ability to distinguish patients who experience an event by time *t* from those who do not is shown as a separate curve. Higher area under the curve (AUC) indicates better discrimination.

In [ ]:
learner.plot_roc()

### Area under the ROC curve (ROC AUC) over time

In [ ]:
learner.plot_metric_time(metric="auc")

### Brier score over time

In [ ]:
learner.plot_metric_time(metric="brier")

### SHAP feature attributions (best model)

SHAP values (Lundberg & Lee 2017) are computed for the best model only — they are expensive (~1 minute) and most informative for the top performer. `importance(method='shap')` returns a global ranking (mean |SHAP| per feature) and, as a side effect, stores the per-patient values on `result.evaluation.shap_values` — exactly what the beeswarm below reads. Unlike permutation importance (which shows *how much* a feature matters), SHAP shows *direction*: whether high feature values push predictions toward higher or lower risk.

In [ ]:
# Compute SHAP now (deferred above): returns a global ranking and stores
# the per-patient values on the result for the beeswarm below.
learner.importance(learner.best_model, method='shap', max_explain=20)
learner.plot_shap(learner.best_model.name, kind='beeswarm')

### Feature interactions

Permutation importance and SHAP rate features one at a time. A 2D partial dependence plot (PDP) asks whether one feature's effect *depends on* another. Each PDP marginalizes over all other features and traces predicted risk across a grid of two, read off the tuned GBSA model. The rendering follows the pair's types: a **contour** for two numeric features (non-parallel contours signal an interaction) and **one risk curve per category level** for a numeric-by-categorical pair (diverging curves signal an interaction). Axis scaling stays at `'auto'`, which puts right-skewed features (pnodes and progrec) on a log scale so dense regions stay legible.

In [ ]:
# Numeric x numeric -> contour. Non-parallel contours mean the two features
# interact: the effect of one on risk depends on the other's level.
learner.plot_interaction_pdp('pnodes', 'progrec', model='gbsa')

In [ ]:
# Numeric x categorical -> one risk curve per group. Diverging curves mean
# progesterone receptor (progrec) affects risk differently with vs without
# hormone therapy (horTh) -- an interaction.
learner.plot_interaction_pdp('horTh', 'progrec', model='gbsa')

### Patient prediction

The plots above describe the whole cohort. The next three views zoom in on a single patient: why the model scored them as it did, what their predicted survival looks like over time, and how to bundle both into one shareable report.

#### Individual patient explanation

`learner.explain()` shows why the model assigned a particular risk score to a single patient. The waterfall plot breaks the prediction into per-feature SHAP contributions. Each bar shows how much that feature pushed the patient's risk score up (red, adds risk) or down (blue, lowers risk).

In [ ]:
# `patient='high_risk'` is the 90th-percentile shorthand.
explanation = learner.explain(patient='high_risk', model='rsf')
explanation.figure

#### Individual patient survival curve

The waterfall above shows *why* the model assigned a high risk score. This shows *what that risk means over time*: the patient's predicted survival curve S(t|x), with a bootstrap confidence band, a median-survival marker at S=0.5, and milestone probabilities P(t > t).

`learner.predict()` returns a `PredictionResult`; its `.plot()` method draws the curve for one patient. We pass the same `'high_risk'` shorthand used by `explain()` above, resolved here against the model's predictions on the test cohort.

In [ ]:
# Same 'high_risk' shorthand as the waterfall, resolved here against the
# model's predicted risk scores on the test cohort.
pred = learner.predict(
    learner.X_test, models='rsf',
    times=[12, 36, 60],          # 1-, 3-, 5-year milestones (months)
    ci=True, ci_band=True, n_bootstrap=100, random_state=42,
)
pred.plot(patient='high_risk')

#### Full patient report

`learner.patient_report()` bundles everything above for a single patient (the waterfall explanation, the individualized survival curve with its CI band, milestone probabilities, and a nearest-neighbor cohort anchor) into one standalone HTML file. It's the handoff format for sharing a case with a clinician, and takes the same `patient=` shorthand used above.

```python
learner.patient_report(
    learner.X_test, patient='high_risk', model='rsf',
    output_path='output/vig2_patient_high_risk.html',
)
```

## External validation on Rotterdam cohort

External validation is the gold standard for assessing generalizability. We apply the best-performing model (trained on GBSG2, n=686) to the independent Rotterdam breast cancer cohort (n=2982) — a 4x larger dataset from a different institution and era.

This cross-cohort design follows Royston & Altman (2013) and tests whether the model's learned risk patterns transfer to a genuinely new population.

In [ ]:
# Load Rotterdam cohort (2982 patients)
X_rot, y_rot = load_survival_data(
    'data/rotterdam.csv',
    time_col='rtime', event_col='recur',
    drop_cols=['rownames', 'pid', 'dtime', 'death', 'year', 'chemo'],
)

# Rotterdam times are also in days - convert to months to match the GBSG2
# training scale (and the model's learned time axis).
y_rot['time'] = y_rot['time'] / DAYS_PER_MONTH

# Align Rotterdam columns to GBSG2 schema (what the model was trained on)
X_rot = X_rot.rename(columns={
    'er': 'estrec', 'hormon': 'horTh', 'meno': 'menostat',
    'pgr': 'progrec', 'grade': 'tgrade', 'size': 'tsize',
    'nodes': 'pnodes',
})
# Convert categorical size to midpoints (mm) matching GBSG2's continuous tsize
X_rot['tsize'] = X_rot['tsize'].map({'<=20': 10, '20-50': 35, '>50': 75}).astype(float)

print(f"Rotterdam: n={len(X_rot)}, features={list(X_rot.columns)}")
print(f"GBSG2 (train): features={list(X.columns)}")

val_result = learner.validate(X_rot, y_rot, models=learner.best_model, name='Rotterdam',
                              n_bootstrap=100, eval_times=eval_times)

In [ ]:
learner.validation("Rotterdam").plot_metric(source="bootstrap",metric="harrell")

In [ ]:
learner.validation("Rotterdam").plot_survival()

## Generate report

In [ ]:
learner.generate_report('output/vig2_report.html', title='Workflow Tour Report');

## References

- Akiba T, Sano S, Yanase T, Ohta T, Koyama M (2019). Optuna: A next-generation hyperparameter optimization framework. *Proc. 25th ACM SIGKDD Int. Conf. on Knowledge Discovery & Data Mining*, 2623–2631. https://doi.org/10.1145/3292500.3330701
- Breiman L (2001). Random forests. *Machine Learning*, 45(1):5–32. https://doi.org/10.1023/A:1010933404324
- Cox DR (1972). Regression models and life-tables. *Journal of the Royal Statistical Society, Series B*, 34(2):187–220. https://doi.org/10.1111/j.2517-6161.1972.tb00899.x
- Lundberg SM, Lee SI (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems 30 (NeurIPS 2017)*, 4765–4774. arXiv:1705.07874. https://arxiv.org/abs/1705.07874
- Royston P, Altman DG (2013). External validation of a Cox prognostic model: principles and methods. *BMC Medical Research Methodology*, 13:33. https://doi.org/10.1186/1471-2288-13-33
- Schumacher M, Bastert G, Bojar H, et al. (1994). Randomized 2×2 trial evaluating hormonal treatment and the duration of chemotherapy in node-positive breast cancer patients (German Breast Cancer Study Group). *Journal of Clinical Oncology*, 12(10):2086–2093. https://doi.org/10.1200/JCO.1994.12.10.2086